In [6]:
# ── Coleta via Ticketmaster Discovery API (plano gratuito) ───────────
# Chave em developer.ticketmaster.com → "Get your API key"
import os, requests, time
from datetime import datetime, timedelta

TM_KEY = os.environ["TICKETMASTER_API_KEY"]
BASE = "https://app.ticketmaster.com/discovery/v2/events.json"

ARTISTAS = ["", ""]          # vazio = todos os shows de música no Brasil
JANELA_DIAS = 180

ini = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
fim = (datetime.utcnow() + timedelta(days=JANELA_DIAS)).strftime("%Y-%m-%dT%H:%M:%SZ")

# Capacidades e regras que não vêm de API — preencha conforme for aprendendo
LOCAIS = {
    # "Allianz Parque": {"capacidade": 43000, "nominal": True, "facial": True},
}

def buscar(keyword=None):
    eventos, pagina = [], 0
    while True:
        p = {"apikey": TM_KEY, "countryCode": "BR", "classificationName": "Music",
             "startDateTime": ini, "endDateTime": fim, "size": 200, "page": pagina}
        if keyword:
            p["keyword"] = keyword
        r = requests.get(BASE, params=p, timeout=30)
        r.raise_for_status()
        d = r.json()
        eventos += d.get("_embedded", {}).get("events", [])
        if pagina >= d.get("page", {}).get("totalPages", 1) - 1 or pagina >= 4:
            break
        pagina += 1
        time.sleep(0.3)
    return eventos

def converter(e):
    venue = (e.get("_embedded", {}).get("venues") or [{}])[0]
    nome_local = venue.get("name", "")
    extra = LOCAIS.get(nome_local, {})
    precos = e.get("priceRanges") or []
    vendas = e.get("sales", {}).get("public", {})
    presales = e.get("sales", {}).get("presales") or []
    estimativas = [] if extra else ["capacidade", "ingresso_nominal", "validacao_facial"]
    return {
        "artista": (e.get("_embedded", {}).get("attractions") or [{}])[0].get("name", e.get("name")),
        "data_show": e.get("dates", {}).get("start", {}).get("localDate"),
        "cidade": venue.get("city", {}).get("name"),
        "uf": venue.get("state", {}).get("stateCode"),
        "local": nome_local,
        "capacidade": extra.get("capacidade"),
        "tiqueteira": "Ticketmaster",
        "abertura_vendas": (vendas.get("startDateTime") or "")[:10] or None,
        "pre_venda": (presales[0].get("startDateTime") or "")[:10] if presales else None,
        "preco_inteira_min": precos[0].get("min") if precos else None,
        "preco_inteira_max": precos[0].get("max") if precos else None,
        "preco_meia_min": None, "preco_meia_max": None,
        "setores": [],
        "publico_ultimo_show_br": None,
        "ingresso_nominal": extra.get("nominal"),
        "validacao_facial": extra.get("facial"),
        "data_extra": None,
        "status_vendas": e.get("dates", {}).get("status", {}).get("code"),
        "estimativas": estimativas,
        "fonte": e.get("url"),
    }

brutos = []
for a in (ARTISTAS or [None]):
    brutos += buscar(a or None)

vistos, SHOWS = set(), []
for e in brutos:
    if e["id"] in vistos:
        continue
    vistos.add(e["id"])
    SHOWS.append(converter(e))

print(f"{len(SHOWS)} show(s) coletado(s) da Ticketmaster")

C:\Users\gabri\AppData\Local\Temp\ipykernel_13084\1259376299.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ini = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
C:\Users\gabri\AppData\Local\Temp\ipykernel_13084\1259376299.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  fim = (datetime.utcnow() + timedelta(days=JANELA_DIAS)).strftime("%Y-%m-%dT%H:%M:%SZ")


81 show(s) coletado(s) da Ticketmaster


In [ ]:
# ── Ranking de relevância dos artistas com show aberto ──────────────
import pandas as pd

ATTR = "https://app.ticketmaster.com/discovery/v2/attractions.json"
cache = {}

def eventos_futuros(nome):
    if nome in cache:
        return cache[nome]
    r = requests.get(ATTR, params={"apikey": TM_KEY, "keyword": nome, "size": 1}, timeout=30)
    a = (r.json().get("_embedded", {}).get("attractions") or [{}])[0]
    cache[nome] = (a.get("upcomingEvents", {}) or {}).get("_total", 0)
    time.sleep(0.2)
    return cache[nome]

df = pd.DataFrame(SHOWS)
df["eventos_futuros"] = df["artista"].map(eventos_futuros)
df["teto_preco"] = df["preco_inteira_max"].fillna(0)

# normaliza os dois sinais de 0 a 1 e soma com peso
for c in ["eventos_futuros", "teto_preco"]:
    mx = df[c].max() or 1
    df[c + "_n"] = df[c] / mx

df["score_relevancia"] = 0.7 * df["eventos_futuros_n"] + 0.3 * df["teto_preco_n"]
df = df.sort_values("score_relevancia", ascending=False).reset_index(drop=True)
df.insert(0, "ranking", df.index + 1)

TOP = 50
SHOWS = df.head(TOP).drop(columns=[c for c in df.columns if c.endswith("_n")]).to_dict("records")
print(f"Top {len(SHOWS)} por relevância — líder: {SHOWS[0]['artista']}")

In [ ]:
# ── Monitor de Shows e Turnês — Brasil ──────────────────────────────
# Rode a célula: gera shows_AAAA-MM-DD_HHMM.json e .xlsx

import json, pandas as pd
from datetime import datetime
from pathlib import Path

SAIDA = Path("./saidas"); SAIDA.mkdir(exist_ok=True)
AGORA = datetime.now()
VERIFICADO_EM = AGORA.strftime("%Y-%m-%d %H:%M")

CAMPOS = [
    "artista", "data_show", "cidade", "uf", "local", "capacidade",
    "tiqueteira", "abertura_vendas", "pre_venda",
    "preco_inteira_min", "preco_inteira_max", "preco_meia_min", "preco_meia_max",
    "setores", "publico_ultimo_show_br", "ingresso_nominal",
    "validacao_facial", "data_extra", "status_vendas",
    "estimativas", "fonte", "verificado_em",
]



def normalizar(s):
    r = {c: s.get(c) for c in CAMPOS}
    r["verificado_em"] = VERIFICADO_EM
    return r

registros = [normalizar(s) for s in SHOWS]

carimbo = AGORA.strftime("%Y-%m-%d_%H%M")
cam_json = SAIDA / f"shows_{carimbo}.json"
cam_xlsx = SAIDA / f"shows_{carimbo}.xlsx"

with open(cam_json, "w", encoding="utf-8") as f:
    json.dump({"gerado_em": VERIFICADO_EM, "total": len(registros),
               "shows": registros}, f, ensure_ascii=False, indent=2)

df = pd.DataFrame(registros)
plano = [
    {"artista": r["artista"], "data_show": r["data_show"], **s}
    for r in registros for s in (r["setores"] or [])
]
with pd.ExcelWriter(cam_xlsx, engine="openpyxl") as w:
    df.assign(
        setores=df["setores"].apply(lambda x: json.dumps(x, ensure_ascii=False)),
        estimativas=df["estimativas"].apply(lambda x: ", ".join(x or [])),
    ).to_excel(w, sheet_name="Shows", index=False)
    if plano:
        pd.DataFrame(plano).to_excel(w, sheet_name="Precos_por_setor", index=False)

print(f"OK: {cam_json.name} e {cam_xlsx.name} — {len(registros)} show(s)")

OK: shows_2026-10-06_2246.json e shows_2026-10-06_2246.xlsx — 81 show(s)
